# Passport (IRC 7345) Certification Analytics POC — setup and refresh

Run this notebook top to bottom (**Run all**) on Serverless compute. It:

1. Creates the schema and a `raw` volume (if missing)
2. Copies the kit's CSVs into the volume (skipped if they are already there)
3. Loads one Delta table per CSV
4. Runs `sql/passport_rules.sql` to build the rule tables and views
5. Validates: answer-key mismatches must be 0 and `v_threshold_gaps` must return 0 rows — **the notebook stops with an error if either fails**
6. Compares finding counts to the reference run (2026-11-04)
7. Writes `genie_benchmark_answers_<catalog>_<schema>.sql` with your names filled in, ready to paste into Genie benchmarks

Re-run it whenever the CSVs or the rules change. Synthetic data only — no FTI.

**Set the two widgets at the top (catalog, schema) before running.** The other widgets default to the kit's folder layout (this notebook sits at the kit root, next to `data/`, `sql/` and `genie/`).

In [ ]:
dbutils.widgets.text("catalog", "workspace", "1. Catalog")
dbutils.widgets.text("schema", "passport_poc", "2. Schema")
dbutils.widgets.text("data_dir", "./data", "3. CSV folder (kit path or /Volumes/...)")
dbutils.widgets.text("rules_sql", "./sql/passport_rules.sql", "4. Rules SQL file")
dbutils.widgets.text("benchmark_sql", "./genie/genie_benchmark_answers.sql", "5. Benchmark SQL template")

import os, re, shutil

CATALOG = dbutils.widgets.get("catalog").strip()
SCHEMA = dbutils.widgets.get("schema").strip()
for name, val in [("catalog", CATALOG), ("schema", SCHEMA)]:
    if not re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*", val):
        raise ValueError(f"{name} '{val}' must be letters, digits and underscores only")

FQ = f"`{CATALOG}`.`{SCHEMA}`"
VOLUME_DIR = f"/Volumes/{CATALOG}/{SCHEMA}/raw"

def resolve(p):
    p = dbutils.widgets.get(p).strip()
    return p if p.startswith("/") else os.path.abspath(p)

DATA_DIR = resolve("data_dir")
RULES_SQL = resolve("rules_sql")
BENCH_SQL = resolve("benchmark_sql")
print(f"Target:    {CATALOG}.{SCHEMA}")
print(f"CSVs from: {DATA_DIR}")
print(f"Rules:     {RULES_SQL}")
print(f"Benchmark: {BENCH_SQL}")

## 1. Schema, volume and CSVs

In [ ]:
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {FQ}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {FQ}.`raw`")

if not os.path.isdir(DATA_DIR):
    raise FileNotFoundError(f"CSV folder not found: {DATA_DIR}. Set the data_dir widget, "
                            f"or upload the CSVs to {VOLUME_DIR} and set data_dir to that path.")

csvs = sorted(f for f in os.listdir(DATA_DIR) if f.lower().endswith(".csv"))
if not csvs:
    raise FileNotFoundError(f"No .csv files in {DATA_DIR}")

if os.path.abspath(DATA_DIR) != os.path.abspath(VOLUME_DIR):
    for f in csvs:
        shutil.copyfile(os.path.join(DATA_DIR, f), os.path.join(VOLUME_DIR, f))
    print(f"Copied {len(csvs)} CSVs to {VOLUME_DIR}")
else:
    print(f"Using {len(csvs)} CSVs already in {VOLUME_DIR}")

## 2. Load one table per CSV

In [ ]:
loaded = []
for f in csvs:
    t = re.sub(r"[^A-Za-z0-9_]", "_", f[:-4])
    spark.sql(f"""
        CREATE OR REPLACE TABLE {FQ}.`{t}` AS
        SELECT * FROM read_files('{VOLUME_DIR}/{f}', format => 'csv', header => true, inferSchema => true)
    """)
    n = spark.table(f"{FQ}.`{t}`").count()
    loaded.append((t, n))
    print(f"loaded {t:<26} {n:>6} rows")

expected_tables = {"pp_entity","pp_module","pp_csed","pp_transaction","pp_lien","pp_appeals_cdp",
                   "pp_run_params","ref_indicator","ref_indicator_reversal","ref_noncert_mft","ref_threshold"}
missing = expected_tables - {t for t, _ in loaded}
if missing:
    raise RuntimeError(f"Required tables missing from the CSV folder: {sorted(missing)}")

## 3. Build the rule tables and views

In [ ]:
with open(RULES_SQL) as fh:
    sql_text = fh.read()

spark.sql(f"USE CATALOG `{CATALOG}`")
spark.sql(f"USE SCHEMA `{SCHEMA}`")

def statements(text):
    # drop full-line comments and the file's own USE lines, then split on ';' at end of line
    keep = [l for l in text.splitlines()
            if not l.strip().startswith("--") and not re.match(r"\s*USE\s+(CATALOG|SCHEMA)\b", l, re.I)]
    for s in re.split(r";\s*(?:\n|$)", "\n".join(keep)):
        if s.strip():
            yield s.strip()

for i, st in enumerate(statements(sql_text), 1):
    head = " ".join(st.split()[:5])
    try:
        spark.sql(st)
    except Exception as e:
        raise RuntimeError(f"Statement {i} failed ({head} ...): {e}") from None
    print(f"ok  {i:>2}  {head}")

## 4. Validate (stops on failure)

In [ ]:
has_key = "pp_answer_key" in {t for t, _ in loaded}

gaps = spark.sql(f"SELECT * FROM {FQ}.v_threshold_gaps").count()
print(f"v_threshold_gaps rows: {gaps}")

mismatches = None
if has_key:
    mm = spark.sql(f"""
    WITH exp AS (
      SELECT DISTINCT k.tin_token, trim(f) AS finding
      FROM {FQ}.pp_answer_key k
      LATERAL VIEW explode(split(coalesce(k.expected_findings, ''), '[;,|]')) x AS f
      WHERE trim(f) NOT IN ('', 'NONE', 'None', 'none')),
    act AS (SELECT DISTINCT tin_token, finding FROM {FQ}.v_findings)
    SELECT 'MISSED' AS issue, e.tin_token, e.finding FROM exp e LEFT ANTI JOIN act a USING (tin_token, finding)
    UNION ALL
    SELECT 'EXTRA', a.tin_token, a.finding FROM act a LEFT ANTI JOIN exp e USING (tin_token, finding)
    """)
    mismatches = mm.count()
    print(f"answer-key mismatches: {mismatches}")
    if mismatches:
        display(mm.limit(50))
else:
    print("pp_answer_key not loaded - answer-key check skipped (expected on real data)")

problems = []
if gaps: problems.append(f"{gaps} certification year(s) missing from ref_threshold")
if mismatches: problems.append(f"{mismatches} answer-key mismatch(es)")
if problems:
    raise RuntimeError("VALIDATION FAILED: " + "; ".join(problems))
print("VALIDATION PASSED")

## 5. Finding counts vs. the reference run (information only)

In [ ]:
reference = {  # Databricks Free Edition run, 2026-11-04, kit v0.5.2 synthetic data
    "CERTIFIED_BELOW_THRESHOLD_AT_CERT": 9, "CERTIFIED_MODULE_NO_PREDICATE": 9,
    "DECERT_DUE_OVERDUE": 36, "DECERT_DUE_PENDING": 18,
    "INFO_BELOW_THRESHOLD_STILL_CERTIFIED": 9, "INFO_EXPEDITE_INELIGIBLE": 9,
    "MISSED_CERTIFICATION": 52, "MODULE_REVERSAL_DUE_OVERDUE": 9,
    "NEW_ELIGIBLE_MODULE_NOT_CERTIFIED": 9, "REVERSED_LATE": 9, "REVERSED_TIMELY": 9,
}
got = {r["finding"]: r["n"] for r in spark.sql(
    f"SELECT finding, COUNT(*) AS n FROM {FQ}.v_findings GROUP BY finding").collect()}
rows = [(k, got.get(k, 0), reference.get(k, 0),
         "same" if got.get(k, 0) == reference.get(k, 0) else "DIFFERENT")
        for k in sorted(set(got) | set(reference))]
display(spark.createDataFrame(rows, "finding string, this_run long, reference long, status string"))
diff = [r for r in rows if r[3] != "same"]
print("Matches the reference run." if not diff else
      f"{len(diff)} finding count(s) differ from the reference - expected if the data changed, otherwise investigate.")

## 6. Genie benchmark SQL for this catalog and schema

In [ ]:
with open(BENCH_SQL) as fh:
    bench = fh.read()
if "{{catalog}}" not in bench:
    raise ValueError("Benchmark template has no {{catalog}}.{{schema}} placeholders - use the v0.5.3 file")
bench = bench.replace("{{catalog}}", CATALOG).replace("{{schema}}", SCHEMA)
out = f"{VOLUME_DIR}/genie_benchmark_answers_{CATALOG}_{SCHEMA}.sql"
with open(out, "w") as fh:
    fh.write(bench)
print(f"Wrote {out}")
print("Next: create the Genie space (see README), paste genie/genie_instructions.md into Instructions,")
print("and add each BENCHMARK QUESTION from that file with its query as the SQL answer.")